# Week 1: The $3M Question — When Will Your Best Customers Stop Buying?
## Kaplan-Meier Survival Curves & BG/NBD Model

**Masterclass:** [Week 1 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week1_the_3m_question.html)

**Datasets:**
- **CDNOW** — Bundled with `lifetimes` library (Fader & Hardie's original dataset)
- **Transactions** — Bundled with `lifetimes` for KM analysis

**What You'll Build:**
1. Kaplan-Meier survival curves with segment comparison & log-rank test
2. BG/NBD model: P(Alive) for every customer
3. Predicted transactions for the next 30/90 days
4. The recency-frequency paradox demonstrated
5. Customer health tier classification

In [ ]:
# Install required packages (run once)
!pip install lifetimes lifelines pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Load the CDNOW Dataset

The CDNOW dataset is from Fader, Hardie & Lee's 2005 paper *"Counting Your Customers the Easy Way."* It contains transaction records from an online CD store. Bundled with `lifetimes`, so no download needed.

**RFM Column Definitions:**
| Column | Meaning | Unit |
|--------|---------|------|
| `frequency` | Number of **repeat** purchases (total - 1) | count |
| `recency` | Time between first and last purchase | weeks |
| `T` | Time between first purchase and end of observation | weeks |
| `monetary_value` | Average order value (repeat purchases only) | dollars |

> **Critical:** `frequency` = repeat purchases, NOT total. A customer who bought once has frequency = 0.

In [ ]:
from lifetimes.datasets import load_cdnow_summary, load_transaction_data

# Pre-summarized RFM data
cdnow_rfm = load_cdnow_summary()
print(f"Customers: {len(cdnow_rfm):,}")
print(f"Columns: {list(cdnow_rfm.columns)}")
cdnow_rfm.head(10)

In [ ]:
# Quick exploration
print("=== Summary Statistics ===")
print(cdnow_rfm.describe().round(2))

print(f"\nCustomers with 0 repeat purchases: {(cdnow_rfm['frequency']==0).sum():,} ({(cdnow_rfm['frequency']==0).mean():.1%})")
print(f"Median repeat purchases: {cdnow_rfm['frequency'].median():.0f}")
print(f"Max repeat purchases: {cdnow_rfm['frequency'].max():.0f}")

In [ ]:
# Visualize RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].hist(cdnow_rfm['frequency'], bins=30, color='#c45d3e', edgecolor='white', alpha=0.85)
axes[0].set_title('Repeat Purchase Frequency')
axes[0].set_xlabel('Repeat Purchases')

axes[1].hist(cdnow_rfm['recency'], bins=30, color='#264653', edgecolor='white', alpha=0.85)
axes[1].set_title('Recency (weeks)')
axes[1].set_xlabel('Weeks Since Last Purchase')

axes[2].hist(cdnow_rfm.loc[cdnow_rfm['monetary_value']>0, 'monetary_value'],
             bins=30, color='#2d6a4f', edgecolor='white', alpha=0.85)
axes[2].set_title('Avg Order Value (repeat buyers)')
axes[2].set_xlabel('Dollars')

plt.suptitle('CDNOW: RFM Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 2: Kaplan-Meier Survival Curves

KM answers: *"What fraction of customers are still active at time t?"*

We use the raw transaction data (bundled with lifetimes) to compute per-customer tenure and churn status.

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

txns = load_transaction_data()
print(f"Transactions: {len(txns):,} | Customers: {txns['id'].nunique():,}")
print(f"Date range: {txns['date'].min()} to {txns['date'].max()}")

# Per-customer: tenure and churn flag
cutoff = txns['date'].max()
last_purch = txns.groupby('id')['date'].max()
first_purch = txns.groupby('id')['date'].min()

survival_df = pd.DataFrame({
    'tenure': (last_purch - first_purch).dt.days,
    'churned': ((cutoff - last_purch).dt.days > 90).astype(int)
})

# Fit KM
kmf = KaplanMeierFitter()
kmf.fit(survival_df['tenure'], survival_df['churned'], label='All Customers')

fig, ax = plt.subplots(figsize=(10, 6))
kmf.plot_survival_function(ax=ax, color='#264653', linewidth=2)
ax.axhline(y=0.5, color='#c45d3e', linestyle='--', alpha=0.5, label='50% survival')
ax.set_title('Kaplan-Meier: Customer Retention', fontsize=14, fontweight='bold')
ax.set_xlabel('Days Since First Purchase')
ax.set_ylabel('Survival Probability')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Median survival time: {kmf.median_survival_time_:.0f} days")

In [ ]:
# Segment comparison: Heavy vs Light buyers
purchase_counts = txns.groupby('id')['date'].count()
survival_df['segment'] = np.where(
    purchase_counts > purchase_counts.median(), 'Heavy Buyer', 'Light Buyer')

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Heavy Buyer': '#2d6a4f', 'Light Buyer': '#c45d3e'}
for seg in ['Heavy Buyer', 'Light Buyer']:
    mask = survival_df['segment'] == seg
    kmf_s = KaplanMeierFitter()
    kmf_s.fit(survival_df.loc[mask, 'tenure'], survival_df.loc[mask, 'churned'], label=seg)
    kmf_s.plot_survival_function(ax=ax, color=colors[seg], linewidth=2)

ax.set_title('Survival by Purchase Volume', fontsize=14, fontweight='bold')
ax.set_xlabel('Days Since First Purchase')
ax.set_ylabel('Survival Probability')
ax.legend()
plt.tight_layout()
plt.show()

# Log-rank test
h = survival_df[survival_df['segment']=='Heavy Buyer']
l = survival_df[survival_df['segment']=='Light Buyer']
lr = logrank_test(h['tenure'], l['tenure'], h['churned'], l['churned'])
print(f"Log-Rank p-value: {lr.p_value:.6f} ({'Significant' if lr.p_value < 0.05 else 'Not significant'})")

---
## Part 3: BG/NBD Model — P(Alive) and Purchase Predictions

The BG/NBD model simultaneously answers:
1. **Is this customer still alive?** → P(Alive)
2. **How many purchases will they make?** → E[X(t)]

**Assumptions:** While active, customers buy at a Poisson rate (Gamma-distributed across population). After each purchase, probability of "dying" follows a geometric distribution (Beta-distributed across population).

In [ ]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(cdnow_rfm['frequency'], cdnow_rfm['recency'], cdnow_rfm['T'])

print("=== BG/NBD Parameters ===")
for param, val in bgf.params_.items():
    print(f"  {param}: {val:.4f}")

In [ ]:
# P(Alive) and predictions
cdnow_rfm['p_alive'] = bgf.conditional_probability_alive(
    cdnow_rfm['frequency'], cdnow_rfm['recency'], cdnow_rfm['T'])

cdnow_rfm['pred_30d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    4.3, cdnow_rfm['frequency'], cdnow_rfm['recency'], cdnow_rfm['T'])

cdnow_rfm['pred_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    13, cdnow_rfm['frequency'], cdnow_rfm['recency'], cdnow_rfm['T'])

print("=== P(Alive) Distribution ===")
print(cdnow_rfm['p_alive'].describe().round(3))
print(f"\nLikely alive (P>0.5): {(cdnow_rfm['p_alive']>0.5).sum():,} ({(cdnow_rfm['p_alive']>0.5).mean():.1%})")
print(f"Likely dead  (P<0.2): {(cdnow_rfm['p_alive']<0.2).sum():,} ({(cdnow_rfm['p_alive']<0.2).mean():.1%})")

In [ ]:
# Visualize P(Alive)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cdnow_rfm['p_alive'], bins=50, color='#264653', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0.5, color='#c45d3e', linestyle='--', linewidth=2, label='P(Alive)=0.5')
axes[0].set_title('Distribution of P(Alive)')
axes[0].set_xlabel('P(Alive)')
axes[0].legend()

sc = axes[1].scatter(cdnow_rfm['recency'], cdnow_rfm['p_alive'],
    c=cdnow_rfm['frequency'], cmap='RdYlGn', alpha=0.4, s=10)
axes[1].set_title('P(Alive) vs Recency (color=Frequency)')
axes[1].set_xlabel('Recency (weeks)')
axes[1].set_ylabel('P(Alive)')
plt.colorbar(sc, ax=axes[1], label='Frequency')
plt.tight_layout()
plt.show()

---
## Part 4: The Recency-Frequency Paradox

High frequency + low recency is suspicious — they *used to* buy often but stopped. The model captures this nuance.

In [ ]:
# Demonstrate with specific profiles
profiles = pd.DataFrame({
    'Profile': ['A: 20 purchases, last 8mo ago', 'B: 3 purchases, last yesterday',
                'C: 1 purchase 2wks ago', 'D: 50 purchases, last week'],
    'frequency': [19, 2, 0, 49],
    'recency': [52-8*4.3, 0.14, 0.14, 0.14],
    'T': [52, 26, 4, 104],
})
profiles['p_alive'] = bgf.conditional_probability_alive(
    profiles['frequency'], profiles['recency'], profiles['T'])
profiles['pred_30d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    4.3, profiles['frequency'], profiles['recency'], profiles['T'])

print("=== Recency-Frequency Paradox ===")
print(profiles[['Profile','p_alive','pred_30d']].to_string(index=False))

---
## Part 5: Model Validation — Frequency-Recency Matrix

In [ ]:
from lifetimes.plotting import plot_frequency_recency_matrix, plot_probability_alive_matrix

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_frequency_recency_matrix(bgf, T=4.3, ax=axes[0], cmap='YlOrRd')
axes[0].set_title('Expected Purchases (Next 30d)', fontsize=13, fontweight='bold')
plot_probability_alive_matrix(bgf, ax=axes[1], cmap='RdYlGn')
axes[1].set_title('P(Alive)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Customer Health Tiers
Create tiers: **Green** (P>0.7), **Yellow** (0.3-0.7), **Red** (P<0.3). Count customers per tier.

In [ ]:
# EXERCISE 1: Your code here
cdnow_rfm['health'] = pd.cut(cdnow_rfm['p_alive'], bins=[0,0.3,0.7,1.0],
    labels=['Red (Likely Churned)', 'Yellow (At-Risk)', 'Green (Healthy)'])
print("=== Customer Health Tiers ===")
print(cdnow_rfm['health'].value_counts().sort_index())
print("\nAs percentages:")
print(cdnow_rfm['health'].value_counts(normalize=True).sort_index().map('{:.1%}'.format))

### Exercise 2: Top 20 by Predicted 90-Day Purchases
Identify the top 20 customers. What patterns do they share?

In [ ]:
# EXERCISE 2: Your code here
top20 = cdnow_rfm.nlargest(20, 'pred_90d')
print("=== Top 20 by Predicted 90-Day Purchases ===")
print(top20[['frequency','recency','T','p_alive','pred_90d']].to_string())
print(f"\nAvg frequency: {top20['frequency'].mean():.1f}")
print(f"Avg P(Alive): {top20['p_alive'].mean():.3f}")

### Exercise 3: Segment KM by First Order Value
Split customers by first order value (above/below median). Do high-value first orders predict longer lifespans?

In [ ]:
# EXERCISE 3: Skeleton — complete this yourself
# Steps:
#   1. From txns, compute first transaction value per customer
#   2. Split into high/low first-order-value segments
#   3. Fit separate KM curves
#   4. Run logrank_test()
print("Complete this exercise using the KM pattern from Part 2.")

---
## Key Takeaways

1. **KM** gives the survival curve shape — *when* customers typically churn
2. **BG/NBD** gives individual P(Alive) and predictions — *who* to target
3. **Recency-frequency interaction** is non-obvious: always evaluate relative to buying rhythm
4. **Log-rank tests** formally compare survival between segments

**Next:** [Week 2 — Cox PH](https://balaviswanath-analytics.github.io/analytics_masterclass/week2_renewal_cliff.html) to understand *why* customers churn